# 05 — Frontage Assignment & Hybrid Spatial Model
Builds the hybrid spatial model for Oxford Street + arteries: frontage tessellation, building/POI assignment to street segments, and dual clustering (unconstrained typology + spatially-constrained regionalisation).

**Rewrite notes (this version):**
This replaces the first draft of this notebook. That draft produced an output with almost no variation across segments — traced to three compounding issues, all now fixed:
1. `LineString` was used but never imported in the tessellation function (would raise `NameError` on first run).
2. The building-height column was looked up under `"height"`; the actual column from the LIDAR pipeline (HANDOFF-03) is `height_m`. The old lookup silently found nothing, so `mean_height`/`std_dev_height` were dropped from the output with no error — the built-form layer of the "hybrid" model was empty.
3. The POI loading step loaded raw point geometries from `POIS_RAW_GPKG`, then silently overwrote that variable with the aggregated `poi_category_summary.csv` before use — the two have different shapes and the swap wasn't intentional (there was a stray comment acknowledging the wiring was unfinished).

Rather than patch these and risk a fourth silent failure of the same kind, this version **fails loudly** wherever the old version guessed: column names are asserted and printed, not silently multi-guessed, and diagnostic cells report actual variation in each feature *before* clustering runs, so a flat feature is caught immediately rather than discovered after the fact in the clustering output.

In [ ]:
import os
import sys
sys.path.insert(0, "..")
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import LineString, Point, Polygon
from shapely.ops import polygonize
from scipy.spatial import Voronoi
from libpysal.weights import W
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from spopt.region import Skater
from config import (
    NETWORK_CENTRALITY_GPKG,
    BUILDINGS_WITH_HEIGHT_GPKG,
    CLUSTERED_HIGHSTREET_GPKG,
    POIS_RAW_GPKG,
    POI_CATEGORY_SUMMARY_CSV,
    PROJECTED_CRS,
    OUTPUTS_DIR,
    NODE_DECIMALS,
    DISTANCES,
    FRONTAGE_BUFFER_M,
    TESSELLATION_SAMPLE_M,
    POI_ASSIGN_MAX_DIST_M,
    HYBRID_MODEL_N_CLUSTERS,
    FEATURE_COLUMNS,
)

## 1. Load inputs & inspect actual state
`%whos`-style discipline before anything else: print shape, CRS, columns, and a sample row for every input, rather than assuming the previous notebook's schema still holds. This is the step the first draft skipped — every downstream "guess the column name" bug traces back to not looking here first.

In [ ]:
edges = gpd.read_file(NETWORK_CENTRALITY_GPKG)
buildings = gpd.read_file(BUILDINGS_WITH_HEIGHT_GPKG)
poi_summary = pd.read_csv(POI_CATEGORY_SUMMARY_CSV)
pois_raw = gpd.read_file(POIS_RAW_GPKG)

for name, obj in [("edges", edges), ("buildings", buildings), 
                  ("poi_summary", poi_summary), ("pois_raw", pois_raw)]:
    print(f"--- {name} ---")
    print("shape:", obj.shape)
    if hasattr(obj, "crs"):
        print("crs:", obj.crs)
    print("columns:", list(obj.columns))
    print(obj.head(2))
    print()

In [ ]:
# Reproject everything to the working projected CRS now, explicitly,
# rather than relying on each function to do it later.
edges = edges.to_crs(PROJECTED_CRS)
buildings = buildings.to_crs(PROJECTED_CRS)
pois_raw = pois_raw.to_crs(PROJECTED_CRS)

## 2. Resolve identifiers — fail loudly, don't guess
Three things need to be pinned down before any geometry work: the edge ID column, the height column, and which POI source is actually usable for a per-point spatial join. Each check below either finds an unambiguous answer or raises an error naming exactly what was found, so a mismatch is caught here rather than propagating silently into a clustering feature that turns out to be all zeros.

In [ ]:
# --- Edge ID column ---
# cityseer's exported edge tables commonly carry the node-pair index under
# a name like this; check known candidates but do NOT fabricate a
# positional fallback without saying so loudly, since a positional index
# is not stable across separate loads/joins.
_edge_id_candidates = ["ns_nodes_idx", "edge_id", "fid", "id"]
EDGE_COL = next((c for c in _edge_id_candidates if c in edges.columns), None)
if EDGE_COL is None:
    edges = edges.reset_index()
    if "index" in edges.columns:
        EDGE_COL = "index"
    print(
        "WARNING: none of", _edge_id_candidates, "found in edges.columns "
        f"({list(edges.columns)}). Falling back to positional index as "
        f"'{EDGE_COL}'. This is NOT safe if edges is reloaded or reordered "
        "between cells/sessions -- confirm this is acceptable before "
        "trusting any join downstream."
    )
else:
    print(f"EDGE_COL resolved to '{EDGE_COL}'")

edges[EDGE_COL] = edges[EDGE_COL].astype(str)
assert edges[EDGE_COL].is_unique, (
    f"'{EDGE_COL}' is not unique in edges -- every downstream groupby/merge "
    "keyed on this column will silently collapse rows. Fix the source data "
    "before continuing."
)

In [ ]:
# --- Height column ---
_height_candidates = ["height_m", "height", "mean_height"]
HEIGHT_COL = next((c for c in _height_candidates if c in buildings.columns), None)
assert HEIGHT_COL is not None, (
    f"No height column found in buildings (checked {_height_candidates}). "
    f"Actual columns: {list(buildings.columns)}. Per HANDOFF-03 this should "
    "be 'height_m' -- check BUILDINGS_WITH_HEIGHT_GPKG was regenerated "
    "correctly before proceeding."
)
print(f"HEIGHT_COL resolved to '{HEIGHT_COL}'")

buildings[HEIGHT_COL] = pd.to_numeric(buildings[HEIGHT_COL], errors="coerce")
n_missing_height = buildings[HEIGHT_COL].isna().sum()
print(f"{n_missing_height}/{len(buildings)} buildings have no height "
      f"({n_missing_height/len(buildings):.1%}) -- expect ~19.5% per "
      "HANDOFF-03 (7906/40461 outside LIDAR tile coverage). If this is "
      "very different, the join upstream may have changed.")

In [ ]:
# --- POI source ---
# Prefer the raw point dataset for a per-point spatial join; the category
# summary CSV is (per its name) very likely pre-aggregated and NOT a
# per-point table, which is what broke the previous draft. Check rather
# than assume either way.
print("pois_raw geometry types:", pois_raw.geom_type.value_counts().to_dict())
print("pois_raw columns:", list(pois_raw.columns))
print()
print("poi_summary columns:", list(poi_summary.columns))
print("poi_summary shape:", poi_summary.shape)
print("poi_summary row sample:")
print(poi_summary.head(3))

_point_like = pois_raw.geom_type.isin(["Point", "MultiPoint"]).all()
_cat_candidates = [c for c in pois_raw.columns if "cat" in c.lower() or "group" in c.lower() or "type" in c.lower()]
assert _point_like, (
    "pois_raw is not all Point/MultiPoint geometry -- calculate_poi_metrics "
    "needs one row per POI with a point location. Inspect POIS_RAW_GPKG."
)
assert len(_cat_candidates) > 0, (
    f"No category-like column found in pois_raw (columns: {list(pois_raw.columns)}). "
    "Need a column identifying shop/amenity/tourism/leisure category per POI "
    "to compute Shannon entropy and the vacant/anchor rates."
)
POI_CAT_COL = _cat_candidates[0]
print(f"\nUsing pois_raw as the POI source, category column '{POI_CAT_COL}'.")
print(f"NOTE: poi_summary ({POI_CATEGORY_SUMMARY_CSV}) is NOT used for the "
      "per-segment join below -- if it's meant to supply the canonical "
      "cluster label taxonomy instead of pois_raw's own category column, "
      "that needs to be joined in explicitly, not assumed.")

## 3. Frontage tessellation (Voronoi)
Thiessen partition of the street network: each edge gets a "frontage zone" of the space nearest to it, bounded by `FRONTAGE_BUFFER_M`. This keeps the DIY-Voronoi approach from the first draft (not `momepy.Tessellation` as originally sketched in HANDOFF-03 — flagging that divergence again here since it wasn't a deliberate methodology decision, just what got built) but fixes the missing import and adds a coverage check, since Voronoi ridges reaching the convex hull boundary are dropped by `polygonize`, which risks under-covering the two ends of the corridor.

In [ ]:
def densify_geometry(line, sample_dist=TESSELLATION_SAMPLE_M):
    """Sample points along a LineString at fixed intervals."""
    distances = np.arange(0, line.length, sample_dist)
    if len(distances) == 0:
        distances = [0.0]
    points = [line.interpolate(d) for d in distances]
    points.append(line.interpolate(line.length))
    return points

def generate_street_tessellation(edges_gdf, edge_id_col=EDGE_COL, 
                                  sample_dist=TESSELLATION_SAMPLE_M, 
                                  max_buffer=FRONTAGE_BUFFER_M):
    """Voronoi tessellation of frontage zones around street edges."""
    sample_points, edge_ids = [], []
    for _, row in edges_gdf.iterrows():
        pts = densify_geometry(row.geometry, sample_dist=sample_dist)
        sample_points.extend(pts)
        edge_ids.extend([row[edge_id_col]] * len(pts))

    coords = np.array([(p.x, p.y) for p in sample_points])
    vor = Voronoi(coords)

    finite_ridges = [
        vor.vertices[idx] for idx in vor.ridge_vertices if -1 not in idx
    ]
    shapely_lines = [LineString(pts) for pts in finite_ridges]
    all_polygons = list(polygonize(shapely_lines))

    voronoi_gdf = gpd.GeoDataFrame(geometry=all_polygons, crs=edges_gdf.crs)
    points_gdf = gpd.GeoDataFrame(
        {edge_id_col: edge_ids}, geometry=sample_points, crs=edges_gdf.crs
    )

    voronoi_assigned = gpd.sjoin(
        voronoi_gdf, points_gdf, how="inner", predicate="intersects"
    )
    tessellation_gdf = voronoi_assigned.dissolve(by=edge_id_col).reset_index()

    street_buffer = edges_gdf.copy()
    street_buffer["geometry"] = street_buffer.geometry.buffer(max_buffer)
    tessellation_gdf = gpd.overlay(
        tessellation_gdf, street_buffer[[edge_id_col, "geometry"]], 
        how="intersection",
    )
    tessellation_gdf = tessellation_gdf[
        tessellation_gdf[f"{edge_id_col}_1"] == tessellation_gdf[f"{edge_id_col}_2"]
    ].copy()
    tessellation_gdf.rename(columns={f"{edge_id_col}_1": edge_id_col}, inplace=True)
    return tessellation_gdf[[edge_id_col, "geometry"]]

In [ ]:
tessellation = generate_street_tessellation(edges, edge_id_col=EDGE_COL)
print(f"{len(tessellation)} tessellation cells for {edges[EDGE_COL].nunique()} edges "
      f"({tessellation[EDGE_COL].nunique()} distinct edges covered).")

# Coverage check: buildings within the corridor buffer that DON'T fall
# inside any tessellation cell. This is exactly the failure mode the
# dropped-unbounded-ridge issue would produce -- check it rather than
# assume it away.
corridor_extent = edges.buffer(FRONTAGE_BUFFER_M).unary_union
tess_extent = tessellation.unary_union
buildings_in_corridor = buildings[buildings.geometry.intersects(corridor_extent)]
uncovered = buildings_in_corridor[~buildings_in_corridor.geometry.intersects(tess_extent)]
print(f"{len(uncovered)}/{len(buildings_in_corridor)} buildings within the corridor "
      f"buffer are NOT covered by any tessellation cell "
      f"({len(uncovered)/max(len(buildings_in_corridor),1):.1%}).")
if len(uncovered) > 0:
    print("Bounding box of uncovered buildings (check if clustered at corridor ends):")
    print(uncovered.total_bounds)

## 4. Assign buildings to frontage zones
Uses the explicit `HEIGHT_COL` resolved above -- no more multi-candidate guessing.

In [ ]:
def assign_buildings_to_frontages(buildings_gdf, tessellation_gdf, edges_gdf, 
                                    edge_id_col=EDGE_COL, height_col=HEIGHT_COL):
    bldg = buildings_gdf.copy()
    bldg["bldg_id"] = bldg.index
    bldg["orig_area"] = bldg.geometry.area
    intersections = gpd.overlay(
        bldg, tessellation_gdf[[edge_id_col, "geometry"]], how="intersection"
    )
    intersections["overlap_area"] = intersections.geometry.area
    assigned = intersections.sort_values("overlap_area", ascending=False).drop_duplicates("bldg_id")
    metrics = assigned.groupby(edge_id_col).agg(
        building_count=("bldg_id", "count"),
        mean_height=(height_col, "mean"),
        std_dev_height=(height_col, "std"),
        total_footprint_area=("orig_area", "sum"),
    ).reset_index()
    metrics["std_dev_height"] = metrics["std_dev_height"].fillna(0.0)
    out = edges_gdf.merge(metrics, on=edge_id_col, how="left")
    out["building_count"] = out["building_count"].fillna(0)
    out["mean_height"] = out["mean_height"].fillna(0)
    out["std_dev_height"] = out["std_dev_height"].fillna(0)
    out["edge_length_m"] = out.geometry.length
    out["frontage_density_100m"] = (out["building_count"] / out["edge_length_m"]) * 100.0
    return assigned, out

assigned_bldgs, edges_with_frontage = assign_buildings_to_frontages(buildings, tessellation, edges)
print(edges_with_frontage[["building_count", "mean_height", "std_dev_height", 
                            "frontage_density_100m"]].describe())
n_zero_bldg = (edges_with_frontage["building_count"] == 0).sum()
print(f"\n{n_zero_bldg}/{len(edges_with_frontage)} edges have zero assigned buildings "
      f"({n_zero_bldg/len(edges_with_frontage):.1%}).")

In [ ]:
_intermediate_path = os.path.join(OUTPUTS_DIR, "edges_with_frontage_metrics.gpkg")
edges_with_frontage.to_file(_intermediate_path, driver="GPKG")
print(f"Saved intermediate: {_intermediate_path}")

## 5. POI metrics (density, entropy, vacant/anchor rates)
Uses `pois_raw` (resolved above) rather than the aggregated summary CSV.

In [ ]:
def calculate_poi_metrics(edges_gdf, pois_gdf, category_col, 
                            edge_id_col=EDGE_COL, max_dist=POI_ASSIGN_MAX_DIST_M):
    pois_with_edges = gpd.sjoin_nearest(
        pois_gdf.to_crs(edges_gdf.crs),
        edges_gdf[[edge_id_col, "geometry"]],
        max_distance=max_dist, how="inner", distance_col="dist_to_edge",
    )
    print(f"{len(pois_with_edges)}/{len(pois_gdf)} POIs matched to an edge within "
          f"{max_dist}m ({len(pois_with_edges)/len(pois_gdf):.1%}).")
    category_counts = (
        pois_with_edges.groupby([edge_id_col, category_col]).size().unstack(fill_value=0)
    )
    total_pois = category_counts.sum(axis=1)
    proportions = category_counts.div(total_pois, axis=0)
    entropy_matrix = np.where(proportions > 0, proportions * np.log(proportions), 0)
    shannon_entropy = -entropy_matrix.sum(axis=1)

    vacant_cols = [c for c in category_counts.columns if "vacant" in str(c).lower()]
    anchor_cols = [c for c in category_counts.columns 
                   if "department" in str(c).lower() or "anchor" in str(c).lower()]
    vacant_counts = category_counts[vacant_cols].sum(axis=1) if vacant_cols else pd.Series(0, index=category_counts.index)
    anchor_counts = category_counts[anchor_cols].sum(axis=1) if anchor_cols else pd.Series(0, index=category_counts.index)

    poi_summary_out = pd.DataFrame({
        edge_id_col: category_counts.index.astype(str),
        "poi_count": total_pois.values,
        "shannon_entropy": shannon_entropy.values,
        "vacant_count": vacant_counts.values,
        "anchor_count": anchor_counts.values,
    })
    edges_poi = edges_gdf.merge(poi_summary_out, on=edge_id_col, how="left").fillna(
        {"poi_count": 0, "shannon_entropy": 0.0, "vacant_count": 0, "anchor_count": 0}
    )
    length_m = edges_poi.geometry.length
    edges_poi["poi_density_100m"] = (edges_poi["poi_count"] / length_m) * 100.0
    edges_poi["vacant_rate_100m"] = (edges_poi["vacant_count"] / length_m) * 100.0
    edges_poi["anchor_rate_100m"] = (edges_poi["anchor_count"] / length_m) * 100.0
    return edges_poi

edges_full = calculate_poi_metrics(edges_with_frontage, pois_raw, category_col=POI_CAT_COL)
print(edges_full[["poi_count", "shannon_entropy", "vacant_rate_100m", 
                   "anchor_rate_100m"]].describe())

## 6. Feature matrix, incl. `movement_conflict_index`
The old version picked whichever betweenness/closeness column happened to sort first among the six `DISTANCES` scales -- an arbitrary pairing, not a deliberate one. This version explicitly pairs **short-range closeness** (local/destination accessibility) against **long-range betweenness** (strategic through-movement) at the min/max of `config.DISTANCES`, matching the "which segments carry through-traffic they don't locally serve" framing from before. Column names are discovered by pattern match against the actual distance value and asserted to exist -- no silent `[0]`-of-a-list picking.

In [ ]:
SHORT_DIST = min(DISTANCES)
LONG_DIST = max(DISTANCES)

def _find_metric_col(cols, metric_keyword, distance):
    matches = [c for c in cols if metric_keyword in c.lower() and str(distance) in c]
    return matches

close_matches = _find_metric_col(edges_full.columns, "closeness", SHORT_DIST)
bet_matches = _find_metric_col(edges_full.columns, "betweenness", LONG_DIST)
print("closeness columns containing", SHORT_DIST, ":", close_matches)
print("betweenness columns containing", LONG_DIST, ":", bet_matches)
print("All centrality-like columns present:", 
      [c for c in edges_full.columns if "closeness" in c.lower() or "betweenness" in c.lower()])

assert len(close_matches) >= 1, (
    f"No closeness column found matching distance {SHORT_DIST} in edges_full. "
    "Check the actual column naming from the centrality notebook and adjust "
    "_find_metric_col's match logic above -- don't guess a prefix blind."
)
assert len(bet_matches) >= 1, (
    f"No betweenness column found matching distance {LONG_DIST} in edges_full. "
    "Same as above."
)
if len(close_matches) > 1 or len(bet_matches) > 1:
    print("WARNING: multiple matches found, using the first -- confirm this is "
          "the intended column (e.g. angular vs metric variant).")

CLOSE_COL, BET_COL = close_matches[0], bet_matches[0]
print(f"\nUsing '{BET_COL}' (long-range, {LONG_DIST}m) / "
      f"'{CLOSE_COL}' (short-range, {SHORT_DIST}m) for movement_conflict_index.")

In [ ]:
def prepare_feature_matrix(edges_gdf, feature_cols, bet_col=BET_COL, close_col=CLOSE_COL):
    edges_gdf = edges_gdf.copy()
    eps = 1e-6
    edges_gdf["movement_conflict_index"] = edges_gdf[bet_col] / (edges_gdf[close_col] + eps)

    available_cols = [c for c in feature_cols if c in edges_gdf.columns]
    missing_cols = [c for c in feature_cols if c not in edges_gdf.columns]
    if missing_cols:
        print(f"WARNING: feature columns not found and excluded: {missing_cols}")

    X_raw = edges_gdf[available_cols].copy()
    X_clean = X_raw.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    # Variation diagnostic -- catch a flat feature HERE, before clustering.
    print("Per-feature variation check:")
    for c in available_cols:
        n_zero = (X_clean[c] == 0).sum()
        print(f"  {c:28s} std={X_clean[c].std():.4f}  "
              f"zero={n_zero}/{len(X_clean)} ({n_zero/len(X_clean):.1%})")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_clean)
    return edges_gdf, X_clean, X_scaled, available_cols

edges_feat, X_clean, X_scaled, final_feature_cols = prepare_feature_matrix(
    edges_full, FEATURE_COLUMNS
)

## 7. Hybrid clustering — unconstrained typology (KMeans) + spatially-constrained regionalisation (SKATER)
The old contiguity graph was built with `libpysal.weights.Rook` directly on the LineString edges. Rook contiguity is defined by shared *boundary segments*, which street-network edges essentially never have -- they share single endpoint *points* at junctions. Depending on the libpysal version this either silently produces a near-empty/disconnected graph (SKATER then fails or returns degenerate regions) or falls back to something libpysal isn't really designed for.

Since the project already has a shared-endpoint adjacency convention (`NODE_DECIMALS`-rounded coordinate matching, used in `network_tracing.py` for the angular tracing), it's more robust to reuse that same convention here directly: two edges are neighbours if they share a rounded endpoint coordinate. That's both more correct for a street network and consistent with the rest of the pipeline's methodology.

In [ ]:
def build_endpoint_adjacency(edges_gdf, edge_id_col=EDGE_COL, decimals=NODE_DECIMALS):
    """Two edges are neighbours iff they share a rounded endpoint coordinate."""
    endpoint_to_edges = {}
    for idx, row in edges_gdf.iterrows():
        line = row.geometry
        for pt in (Point(line.coords[0]), Point(line.coords[-1])):
            key = (round(pt.x, decimals), round(pt.y, decimals))
            endpoint_to_edges.setdefault(key, set()).add(row[edge_id_col])

    neighbors = {eid: set() for eid in edges_gdf[edge_id_col]}
    for eid_set in endpoint_to_edges.values():
        for eid in eid_set:
            neighbors[eid] |= (eid_set - {eid})

    # index-position based, matching edges_gdf row order (spopt/libpysal expect this)
    id_list = list(edges_gdf[edge_id_col])
    id_to_pos = {eid: i for i, eid in enumerate(id_list)}
    neighbors_pos = {
        id_to_pos[eid]: [id_to_pos[n] for n in neighbors[eid]] for eid in id_list
    }

    w = W(neighbors_pos)
    n_islands = len(w.islands)
    print(f"Endpoint-adjacency graph: {w.n} edges, {n_islands} islands "
          f"({n_islands/w.n:.1%}), mean neighbours "
          f"{sum(len(v) for v in neighbors_pos.values())/w.n:.2f}")
    if n_islands / w.n > 0.05:
        print("WARNING: >5% of edges have no neighbours -- check NODE_DECIMALS "
              "tolerance and whether edges_gdf geometries were snapped/simplified "
              "after the centrality export (coordinates may no longer match exactly).")
    return w

def run_hybrid_clustering(edges_gdf, X_scaled, feature_cols, n_clusters=HYBRID_MODEL_N_CLUSTERS):
    edges_out = edges_gdf.reset_index(drop=True).copy()
    print("--- KMeans (unconstrained typology) ---")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    edges_out["kmeans_cluster"] = kmeans.fit_predict(X_scaled)
    print(edges_out["kmeans_cluster"].value_counts().sort_index())

    print("\n--- SKATER (spatially-constrained regionalisation) ---")
    w_graph = build_endpoint_adjacency(edges_out)
    skater_model = Skater(
        edges_out, w_graph, attrs_name=feature_cols, n_clusters=n_clusters, floor=1,
    )
    skater_model.solve()
    edges_out["skater_region"] = skater_model.labels_
    print(edges_out["skater_region"].value_counts().sort_index())
    return edges_out, kmeans, skater_model

clustered_edges, kmeans_model, skater_model = run_hybrid_clustering(
    edges_feat, X_scaled, final_feature_cols
)

## 8. Save + final variation sanity check
Per-cluster feature means -- if clusters aren't differentiating on a given feature, this is where it shows up immediately, rather than being discovered later in the write-up.

In [ ]:
print("KMeans cluster profiles (mean of each feature):")
print(clustered_edges.groupby("kmeans_cluster")[final_feature_cols].mean().round(3))
print("\nSKATER region profiles (mean of each feature):")
print(clustered_edges.groupby("skater_region")[final_feature_cols].mean().round(3))

In [ ]:
clustered_edges.to_file(CLUSTERED_HIGHSTREET_GPKG, driver="GPKG")
print(f"Saved: {CLUSTERED_HIGHSTREET_GPKG}")

## Open items to confirm
- **`_edge_id_candidates` / `_height_candidates` / `_cat_candidates`**: these lists are best-guesses at plausible names based on HANDOFF docs; the assertions will tell you immediately if they're wrong, but the *correct* name still needs eyeballing against the printed diagnostics in Section 1 the first time this runs.
- **`poi_summary` / `POI_CATEGORY_SUMMARY_CSV` is currently unused** in the clustering pipeline. If it's meant to supply a canonical category-cluster taxonomy (the "10 shop clusters / 8 amenity clusters" from the POI pipeline) rather than `pois_raw`'s raw OSM category column, that mapping needs to be joined onto `pois_raw` explicitly before `calculate_poi_metrics` runs -- right now this notebook computes entropy over whatever `pois_raw`'s own category column contains, which may be finer- or coarser-grained than the manual clusters from HANDOFF-02/03.
- **Voronoi vs `momepy.Tessellation`**: still not resolved as a deliberate methodology choice, just flagging again since it affects the writeup framing.
- **Contiguity graph**: `NODE_DECIMALS` (3 dp, i.e. mm-scale at EPSG:32630) is reused here from `network_tracing.py`'s convention on the assumption that `edges_gdf`'s geometries are the same topologically-cleaned network used for the angular tracing. If `NETWORK_CENTRALITY_GPKG` went through any additional simplification after that stage, endpoints may no longer match at this tolerance -- the island-count warning in Section 7 is the check for this.